# Power Profiling — YOLO26 Edge Devices

**Objective:** Collect USB power draw from the JT-TC66C power meter for each
(model, resolution) condition and merge with the existing Table I timing + MOT data
to produce a complete per-device power profile.

**Protocol (noise reduction):**
- **Warm-up:** 120 s per repetition (recorded but discarded in analysis)
- **Measurement window:** 180 s per repetition (trimmed 5 s each end)
- **Repetitions:** 5 runs per condition (CV < 0.3 %, 95 % CI ±0.35 %)
- **Reporting:** mean ± 95 % CI across runs (median + IQR as robust alternative)

**Prerequisites:**
- TC66C connected between USB-C power supply and the DUT
- `DEVICE_PROFILE` env var set to the **target device** profile (e.g., `jetson_nano.yaml`)

## Connection

The notebook auto-detects the TC66C transport:

1. **BLE** (primary) — TC66C Bluetooth menu ON (Menu 6), no phone/app connected
2. **USB serial** (fallback) — TC66C micro-USB data port connected to this machine (`/dev/ttyACM0`)
3. **Error** — if neither is available, a clear message is shown

```
┌──────────────────────┐  BLE or serial  ┌─────────┐      USB-C      ┌─────────┐
│  This notebook       │ ◄────────────── │  TC66C  │ ◄────────────── │   DUT   │
│  - detects TC66C     │                 │(power   │                 │(RPi/    │
│  - logs power to CSV │                 │ meter)  │                 │ Jetson) │
└──────────────────────┘                 └─────────┘                 └─────────┘
```

In [ ]:
import asyncio
import os
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import nest_asyncio

from benchmark.config import DATA_ROOT, RESULTS_RAW, SEQUENCES, SEQ_SUFFIX, IMGSZ_BASE
from benchmark.device_profile import load_profile, apply_thread_pinning
from benchmark.tc66c import (
    collect, collect_serial, scan_for_tc66c,
    summarise_readings, trim_warmup, summarise_across_runs,
)

# Patch event loop so asyncio.run() works inside Jupyter
nest_asyncio.apply()

PROFILE = load_profile(os.environ.get("DEVICE_PROFILE"))
apply_thread_pinning(PROFILE)

# ── Output paths ────────────────────────────────────────────────────────────
POWER_DIR     = RESULTS_RAW.parent / "power" / PROFILE.result_tag
PROFILING_DIR = RESULTS_RAW.parent / "profiling"
FIGURES_DIR   = RESULTS_RAW.parent / "figures"
POWER_DIR.mkdir(parents=True, exist_ok=True)

# ── Measurement parameters ─────────────────────────────────────────────────
IDLE_DURATION_S    = 30.0    # idle baseline window (low-variance, single measurement)
WARMUP_DURATION_S  = 120.0   # warm-up phase per rep (recorded but discarded)
MEASURE_DURATION_S = 180.0   # measurement window per rep (after warm-up)
N_REPS             = 5       # repetitions per condition (CV < 0.3%, CI95 ±0.35%)
INTERVAL_S         = 1.0     # TC66C poll interval
TRIM_S             = 5.0     # seconds trimmed from each end of measurement window
SKIP_EXISTING      = True

# Total collection time per repetition (warm-up + measurement)
TOTAL_REP_S = WARMUP_DURATION_S + MEASURE_DURATION_S

# ── IEEE two-column figure style ──────────────────────────────────────────
matplotlib.rcParams.update({
    "font.size": 10, "axes.titlesize": 10, "axes.labelsize": 10,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "lines.linewidth": 1.2, "axes.linewidth": 0.8,
})

print(f"Device      : {PROFILE.device_label}")
print(f"Backend     : {PROFILE.backend}")
print(f"Tag         : {PROFILE.result_tag}")
print(f"Power dir   : {POWER_DIR}")
print(f"Protocol    : {WARMUP_DURATION_S:.0f}s warm-up + {MEASURE_DURATION_S:.0f}s measure × {N_REPS} reps")

In [ ]:
# ── TC66C transport detection ────────────────────────────────────────────
# Priority: BLE (primary) → USB serial (fallback) → error
# The chosen transport is wrapped in `power_collect()` so downstream cells
# don't need to know which transport is active.

import glob as _glob

TRANSPORT = None       # "ble" | "serial"
TC66C_ADDRESS = None   # BLE MAC (when TRANSPORT == "ble")
TC66C_PORT = None      # serial device path (when TRANSPORT == "serial")

# 1. Try BLE
try:
    TC66C_ADDRESS = asyncio.run(scan_for_tc66c(timeout=10.0))
    TRANSPORT = "ble"
    print(f"TC66C found via BLE: {TC66C_ADDRESS}")
except Exception as ble_err:
    print(f"BLE scan failed: {ble_err}")

# 2. Fallback to USB serial
if TRANSPORT is None:
    serial_candidates = sorted(_glob.glob("/dev/ttyACM*"))
    for port in serial_candidates:
        try:
            import serial as _pyserial
            ser = _pyserial.Serial(port, baudrate=115200, timeout=2)
            import time as _time; _time.sleep(0.2)
            ser.reset_input_buffer()
            ser.write(b"getva\r\n")
            probe = ser.read(192)
            ser.close()
            if len(probe) == 192:
                TC66C_PORT = port
                TRANSPORT = "serial"
                print(f"TC66C found via USB serial: {TC66C_PORT}")
                break
        except Exception:
            continue

if TRANSPORT is None:
    raise RuntimeError(
        "TC66C not found on any transport.\n"
        "  BLE:    ensure TC66C BT menu is ON (Menu 6), no phone connected\n"
        "  Serial: connect TC66C micro-USB data port, check /dev/ttyACM*\n"
        "          user must be in 'dialout' group (sudo usermod -aG dialout $USER)"
    )


def power_collect(duration_s: float, interval_s: float, csv_path: Path = None,
                  on_reading=None, stop_event=None) -> list:
    """Collect TC66C readings using the detected transport (BLE or serial)."""
    if TRANSPORT == "ble":
        return asyncio.run(collect(
            TC66C_ADDRESS, duration_s=duration_s, interval_s=interval_s,
            csv_path=csv_path, on_reading=on_reading, stop_event=stop_event,
        ))
    else:
        return collect_serial(
            TC66C_PORT, duration_s=duration_s, interval_s=interval_s,
            csv_path=csv_path, on_reading=on_reading, stop_event=stop_event,
        )


print(f"\nTransport: {TRANSPORT.upper()}")

In [ ]:
from __future__ import annotations

# ── Condition iterator helpers ───────────────────────────────────────────
# Centralized in device_profile.py — handles hailo, tensorrt, and dynamic backends.
from benchmark.device_profile import iter_model_imgsz

def _model_stem(model_path: str) -> str:
    """Base model name without extension, resolution suffix, or quality tag.
    yolo26n_576_hq.hef -> yolo26n   |   yolo26n_hq.hef -> yolo26n
    yolo26n_576.hef    -> yolo26n   |   yolo26n.pt      -> yolo26n
    """
    stem = model_path.rsplit(".", 1)[0]
    # Strip quality tags before resolution parsing
    for tag in ("_hq", "_lq"):
        if stem.endswith(tag):
            stem = stem[: -len(tag)]
            break
    parts = stem.rsplit("_", 1)
    return parts[0] if len(parts) == 2 and parts[-1].isdigit() else stem

def _power_csv(model_path: str, imgsz: int, rep: int | None = None) -> Path:
    """CSV path for a single repetition or legacy single-run file.
    rep=None  → results/power/{tag}/{stem}_{imgsz}.csv       (legacy)
    rep=3     → results/power/{tag}/{stem}_{imgsz}_rep03.csv  (multi-rep)
    """
    stem = _model_stem(model_path)
    if rep is not None:
        return POWER_DIR / f"{stem}_{imgsz}_rep{rep:02d}.csv"
    return POWER_DIR / f"{stem}_{imgsz}.csv"

def _rep_csvs(model_path: str, imgsz: int) -> list[Path]:
    """All existing repetition CSVs for a condition, sorted by rep index."""
    stem = _model_stem(model_path)
    pattern = f"{stem}_{imgsz}_rep*.csv"
    return sorted(POWER_DIR.glob(pattern))

def _condition_done(model_path: str, imgsz: int) -> bool:
    """True if all N_REPS repetition CSVs exist for this condition."""
    return len(_rep_csvs(model_path, imgsz)) >= N_REPS

conditions = list(iter_model_imgsz(PROFILE))
print(f"{len(conditions)} conditions × {N_REPS} reps = {len(conditions) * N_REPS} total measurements\n")
for mp, sz in conditions:
    n_done = len(_rep_csvs(mp, sz))
    status = f"DONE ({n_done}/{N_REPS})" if n_done >= N_REPS else f"{n_done}/{N_REPS}"
    print(f"  {_model_stem(mp):12s}  {sz}px  [{status}]")

In [ ]:
# ── Idle baseline measurement ────────────────────────────────────────────
# 30 s with no inference running.  Saved once, reused on subsequent runs.
# Operator: ensure the DUT is idle before running this cell.

IDLE_CSV = POWER_DIR / "idle.csv"

if IDLE_CSV.exists() and SKIP_EXISTING:
    print(f"Idle baseline loaded from {IDLE_CSV}")
else:
    print(f"Collecting idle baseline (30 s) via {TRANSPORT.upper()} ...")
    readings = power_collect(
        duration_s=IDLE_DURATION_S,
        interval_s=INTERVAL_S,
        csv_path=IDLE_CSV,
        on_reading=lambda r: print(f"  {r.power_W:.3f} W", flush=True),
    )
    print(f"Done — {len(readings)} samples saved to {IDLE_CSV}")

idle_stats = summarise_readings(pd.read_csv(IDLE_CSV), trim_s=TRIM_S)
IDLE_MEAN_W = idle_stats["mean_W"]
print(f"Idle: mean={IDLE_MEAN_W:.3f} W  std={idle_stats['std_W']:.3f} W  "
      f"peak={idle_stats['peak_W']:.3f} W  ({idle_stats['n_samples']} samples)")

In [ ]:
# ── On-device measurement ─────────────────────────────────────────────────
# Power logging runs in a background thread (via power_collect) while
# inference runs in the main thread.  Both share a threading.Event for
# clean shutdown.
#
# Each condition is measured N_REPS times.  Per repetition:
#   1. Power thread collects for WARMUP_DURATION_S + MEASURE_DURATION_S
#   2. Inference runs for the same duration (time-capped)
#   3. Warm-up portion is discarded during analysis (cell 7), not here
#
# Only one sequence (MOT17-09, 525 frames) is used as the power workload —
# power draw depends on model + resolution, not on which frames are processed.
# Inference loops continuously until the time budget expires.

import threading
import time as _time

from benchmark.device_profile import try_load_model, resolve_model_path
from benchmark.runner import run_sequence
if PROFILE.backend == "hailo":
    from benchmark.hailo_runner import run_sequence_hailo
if PROFILE.backend == "tensorrt_hq":
    from benchmark.trt_runner import run_sequence_trt

# Disposable directory for inference output during power measurement
INFERENCE_SCRATCH = POWER_DIR / "_scratch"
INFERENCE_SCRATCH.mkdir(parents=True, exist_ok=True)

# Single sequence for power workload (shortest, 525 frames)
POWER_SEQ = "MOT17-09"
POWER_SEQ_DIR = DATA_ROOT / f"{POWER_SEQ}-{SEQ_SUFFIX}"


def _power_thread(csv_path, stop_evt):
    """Background thread: collect power readings until stop_evt or timeout."""
    power_collect(
        duration_s=TOTAL_REP_S + 30,  # extra margin; stop_evt ends it early
        interval_s=INTERVAL_S,
        csv_path=csv_path,
        stop_event=stop_evt,
    )


for model_path, imgsz in conditions:
    stem = _model_stem(model_path)

    if SKIP_EXISTING and _condition_done(model_path, imgsz):
        n_done = len(_rep_csvs(model_path, imgsz))
        print(f"  {stem} {imgsz}px: skip (all {n_done} reps exist)")
        continue

    print(f"\n══ {stem} @ {imgsz}px ══════════════════════════")

    for rep in range(N_REPS):
        csv_path = _power_csv(model_path, imgsz, rep=rep)

        if SKIP_EXISTING and csv_path.exists():
            print(f"  rep {rep:02d}: skip (exists)")
            continue

        print(f"  rep {rep:02d}/{N_REPS-1}: {WARMUP_DURATION_S:.0f}s warm-up + {MEASURE_DURATION_S:.0f}s measure ...")

        # Start power logging in background thread
        stop_evt = threading.Event()
        t = threading.Thread(target=_power_thread, args=(csv_path, stop_evt), daemon=True)
        t.start()
        _time.sleep(2.0)  # allow first sample before inference starts

        # Run inference for the full warm-up + measurement duration
        out_csv = INFERENCE_SCRATCH / f"{POWER_SEQ}_{stem}_{imgsz}_rep{rep:02d}.csv"
        if PROFILE.backend == "hailo":
            resolved = resolve_model_path(model_path)
            run_sequence_hailo(resolved, POWER_SEQ_DIR, imgsz=imgsz, out_csv=out_csv,
                               max_duration_s=TOTAL_REP_S)
        elif PROFILE.backend == "tensorrt_hq":
            resolved = resolve_model_path(model_path)
            run_sequence_trt(resolved, POWER_SEQ_DIR, imgsz=imgsz, out_csv=out_csv,
                             max_duration_s=TOTAL_REP_S)
        else:
            model, err = try_load_model(model_path, PROFILE.torch_device)
            if err:
                print(f"    SKIP — could not load: {err}")
                stop_evt.set()
                t.join(timeout=10)
                continue
            run_sequence(model, POWER_SEQ_DIR, imgsz=imgsz, out_csv=out_csv,
                         max_duration_s=TOTAL_REP_S)
            del model

        # Stop the power logger and wait for thread to finish
        stop_evt.set()
        t.join(timeout=10)

        n_samples = sum(1 for _ in open(csv_path)) - 1 if csv_path.exists() else 0
        print(f"    {n_samples} samples → {csv_path.name}")

    n_done = len(_rep_csvs(model_path, imgsz))
    print(f"  {stem} {imgsz}px: {n_done}/{N_REPS} reps complete")

# Clean up disposable inference output
import shutil
shutil.rmtree(INFERENCE_SCRATCH, ignore_errors=True)
print("\nDone — scratch inference output cleaned up.")

In [ ]:
# ── Aggregate power statistics across repetitions ────────────────────────
# For each condition: load all rep CSVs, discard warm-up, trim edges,
# compute per-run mean, then aggregate across runs (mean ± 95% CI).
# Falls back to legacy single-run CSVs if no rep files exist.

power_rows = []

for model_path, imgsz in conditions:
    stem = _model_stem(model_path)
    rep_files = _rep_csvs(model_path, imgsz)

    # Fallback: legacy single-run CSV (pre-repetition format)
    if not rep_files:
        legacy = _power_csv(model_path, imgsz)
        if legacy.exists():
            rep_files = [legacy]
        else:
            print(f"  WARNING: no power data for {stem} {imgsz}px")
            continue

    # Per-rep statistics: discard warm-up, then trim edges
    run_summaries = []
    for csv_path in rep_files:
        df = pd.read_csv(csv_path)
        df_meas = trim_warmup(df, warmup_s=WARMUP_DURATION_S)
        stats = summarise_readings(df_meas, trim_s=TRIM_S)
        run_summaries.append(stats)

    # Cross-run aggregation
    agg = summarise_across_runs(run_summaries)

    power_rows.append({
        "model":     model_path,
        "imgsz":     imgsz,
        "mean_W":    agg["mean_W"],
        "ci95_W":    agg["ci95_W"],
        "std_W":     agg["std_W"],
        "median_W":  agg["median_W"],
        "iqr_W":     agg["iqr_W"],
        "peak_W":    agg["peak_W"],
        "delta_W":   round(agg["mean_W"] - IDLE_MEAN_W, 4),
        "n_runs":    agg["n_runs"],
    })

power_df = pd.DataFrame(power_rows)
print(f"Power profile — {PROFILE.device_label}  ({N_REPS} reps, 95% CI)\n")
print(power_df[["model", "imgsz", "mean_W", "ci95_W", "median_W", "iqr_W", "delta_W", "n_runs"]]
      .to_string(index=False))

In [ ]:
# ── Merge with Table I and save ──────────────────────────────────────────
# Load existing profiling table, aggregate timing/MOT across sequences
# (power is per-(model, imgsz), not per-sequence), join, compute fps/W.

table1_path = PROFILING_DIR / f"table1_profiling_{PROFILE.result_tag}.csv"

if not table1_path.exists():
    print(f"WARNING: {table1_path.name} not found — run notebook 01 first.")
    print("Saving power-only table instead.")
    out_path = PROFILING_DIR / f"table1_power_{PROFILE.result_tag}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    power_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
else:
    table1 = pd.read_csv(table1_path)

    # Aggregate timing/MOT across sequences
    agg = (
        table1.groupby(["model", "imgsz"])
        .agg(fps=("fps", "mean"), mota=("mota", "mean"), idf1=("idf1", "mean"))
        .reset_index()
        .round({"fps": 1, "mota": 3, "idf1": 3})
    )

    merged = agg.merge(power_df, on=["model", "imgsz"], how="left")
    merged["fps_per_W"] = (merged["fps"] / merged["mean_W"]).round(2)

    out_path = PROFILING_DIR / f"table1_power_{PROFILE.result_tag}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False)

    print(f"Saved: {out_path}\n")
    print(merged[["model", "imgsz", "fps", "mota", "mean_W", "ci95_W", "delta_W", "fps_per_W", "n_runs"]]
          .to_string(index=False))

In [ ]:
# ── Figure: power draw vs model complexity ───────────────────────────────
# Error bars show 95% confidence interval across repetitions.

MODEL_ORDER  = ["yolo26n", "yolo26s", "yolo26m", "yolo26l", "yolo26x"]
MODEL_LABELS = {"yolo26n": "N", "yolo26s": "S", "yolo26m": "M",
                "yolo26l": "L", "yolo26x": "X"}
IMGSZ_COLORS = {640: "#1f77b4", 576: "#ff7f0e", 512: "#2ca02c",
                448: "#d62728", 384: "#9467bd", 320: "#8c564b"}

fig, ax = plt.subplots(figsize=(3.5, 2.8))

plot_df = power_df.copy()
plot_df["stem"] = plot_df["model"].apply(_model_stem)

for imgsz, grp in plot_df.groupby("imgsz"):
    grp = grp.set_index("stem").reindex(MODEL_ORDER).dropna(subset=["mean_W"])
    x = range(len(grp))
    ax.errorbar(
        x, grp["mean_W"], yerr=grp["ci95_W"],
        fmt="o-", color=IMGSZ_COLORS.get(imgsz, "grey"),
        label=f"{imgsz}px", markersize=4, capsize=3,
    )

ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER])
ax.set_xlabel("Model variant")
ax.set_ylabel("Power draw (W)")
ax.set_title(f"USB power — {PROFILE.device_label}")
ax.legend(fontsize=7, ncol=2, title="95% CI", title_fontsize=6)
ax.grid(axis="y", linewidth=0.4, alpha=0.5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
for fmt in ("pdf", "png"):
    path = FIGURES_DIR / f"fig_power_{PROFILE.result_tag}.{fmt}"
    fig.savefig(path, bbox_inches="tight", dpi=300)
    print(f"Saved: {path}")
plt.show()

In [ ]:
# ── Figure: inference efficiency (fps / W) ───────────────────────────────
# Requires the merged table from cell 8.

if "merged" not in dir():
    print("Skipped — Table I merge was not available (run notebook 01 first).")
else:
    fig, ax = plt.subplots(figsize=(3.5, 2.8))

    eff_df = merged.copy()
    eff_df["stem"] = eff_df["model"].apply(_model_stem)

    for imgsz, grp in eff_df.groupby("imgsz"):
        grp = grp.set_index("stem").reindex(MODEL_ORDER).dropna(subset=["fps_per_W"])
        x = range(len(grp))
        ax.plot(
            x, grp["fps_per_W"], "s--",
            color=IMGSZ_COLORS.get(imgsz, "grey"),
            label=f"{imgsz}px", markersize=4,
        )

    ax.set_xticks(range(len(MODEL_ORDER)))
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER])
    ax.set_xlabel("Model variant")
    ax.set_ylabel("Efficiency (fps / W)")
    ax.set_title(f"Inference efficiency — {PROFILE.device_label}")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(axis="y", linewidth=0.4, alpha=0.5)

    for fmt in ("pdf", "png"):
        path = FIGURES_DIR / f"fig_efficiency_{PROFILE.result_tag}.{fmt}"
        fig.savefig(path, bbox_inches="tight", dpi=300)
        print(f"Saved: {path}")
    plt.show()

## Observations

- **Protocol:** 120 s warm-up + 180 s measure × 5 reps per condition
- Idle baseline: *(fill after run)*
- Delta-W range: *(fill after run)*
- Mean 95% CI width: *(fill after run — indicates measurement stability)*
- Median vs mean agreement: *(fill — large divergence suggests outliers)*
- Most efficient condition (fps/W): *(fill after run)*

## Next steps

- Repeat for each remaining device profile; CSVs accumulate per `result_tag`
- Run `notebooks/03_results_figures.ipynb` — merge `table1_power_*.csv` into final paper tables
- Add `mean_W`, `ci95_W`, and `fps_per_W` columns to the paper's Table I